# Trabalho de DL - Classificação de Imagens (CNN) - G13

### 1. Importar bibliotecas

In [ ]:
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

In [ ]:
# Garantir a reprodutibilidade
import random
import numpy as np

RAND_SEED = 13 #Em homenagem ao grupo!

random.seed(RAND_SEED)
np.random.seed(RAND_SEED)

torch.manual_seed(RAND_SEED)
torch.cuda.manual_seed(RAND_SEED)
torch.cuda.manual_seed_all(RAND_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

g = torch.Generator()
g.manual_seed(RAND_SEED)

### 2. Macro definições

In [ ]:
# ==========================
# Configurações
# ==========================
BATCH_SIZE = 128
EPOCHS = 20
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print(f"Usando dispositivo: {DEVICE}")

In [ ]:
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(RAND_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

### 3. Transformações

In [ ]:
# ==========================
# Transformações
# ==========================
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2023, 0.1994, 0.2010)
    )
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2023, 0.1994, 0.2010)
    )
])


### 4. Carregar o dataset

In [ ]:

# ==========================
# Dataset e DataLoader
# ==========================

# ==========================
# Dataset completo
# ==========================
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

# Se só houver CPU para rodar, então utiliza apenas 10% da base de treino/teste para agilizar
if DEVICE.type == 'cpu':
    # ====================================================
    # obtem os indices de apenas 10% da base para agilizar
    # ====================================================

    indices = range(len(train_dataset))
    labels = train_dataset.targets

    subset_idx, _ = train_test_split(
        list(indices),
        train_size=0.1,
        stratify=labels,
        random_state=RAND_SEED
    )

    train_subset = Subset(train_dataset, subset_idx)

    indices = range(len(test_dataset))
    labels = test_dataset.targets

    subset_idx, _ = train_test_split(
        list(indices),
        train_size=0.1,
        stratify=labels,
        random_state=RAND_SEED
    )

    test_subset = Subset(test_dataset, subset_idx)
else:
    train_subset = train_dataset
    test_subset = test_dataset


# ================================
# cria loader do treino e do teste
# ================================

train_loader = torch.utils.data.DataLoader(
    #train_dataset,
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=g
)

test_loader = torch.utils.data.DataLoader(
    #test_dataset,
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    generator=g
)

### 5. Definição das Classes

In [ ]:
# Classes do CIFAR-10
classes = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
)

### 6. Definição do modelo (classe)

In [ ]:
# ==========================
# Modelo CNN
# ==========================
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(DEVICE)


### 7. Função de perda e otimizador

In [ ]:
# ==========================
# Função de perda e otimizador
# ==========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

### 8. Treinamento e Avaliação

In [ ]:
# ==========================
# Treinamento e Avaliação
# ==========================
train_acc_history = []
test_acc_history = []
train_loss_history = []

for epoch in range(EPOCHS):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)

    train_acc_history.append(train_acc)
    train_loss_history.append(train_loss)

    # Avaliação
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_acc = 100 * correct / total
    test_acc_history.append(test_acc)

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Loss={train_loss:.4f} "
        f"Train={train_acc:.2f}% "
        f"Test={test_acc:.2f}%"
    )

### 9. Mostrando no gráfico

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.DataFrame({
    "epoch": range(1, EPOCHS + 1),
    "train_loss": train_loss_history,
    "train_acc": train_acc_history,
    "test_acc": test_acc_history
})


plt.figure(figsize=(8, 5))
plt.plot(history["epoch"], history["train_acc"], label="Treino")
plt.plot(history["epoch"], history["test_acc"], label="Teste")

plt.xlabel("Época")
plt.ylabel("Acurácia (%)")
plt.title(f"Acurácia por época usando {DEVICE.type}")
plt.legend()
plt.grid(True)

plt.show()

### 10. Salvar o modelo

In [ ]:
# ==========================
# Salvar modelo
# ==========================
torch.save(model.state_dict(), "cifar10_model.pth")
print("Modelo salvo em cifar10_model.pth")